# **Import Libraries**

In [1]:
# --- Standard Python Libraries ---
import copy
import math
import string
from collections import Counter

# --- Third-Party Libraries ---
import matplotlib.pyplot as plt
import nltk
import numpy as np
from nltk import ngrams
from nltk.corpus import gutenberg
from nltk.corpus import stopwords

# --- NLTK Data Downloads ---
nltk.download('punkt')
nltk.download('gutenberg')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

# **Download Training and Testing Corpus**

In [2]:
training_corpus = gutenberg.sents('shakespeare-caesar.txt') + gutenberg.sents('shakespeare-macbeth.txt')
testing_corpus = gutenberg.sents('shakespeare-hamlet.txt')

# **Defining N-Gram Class**

In [3]:
class NGramModel():
  def __init__(self, corpus, n):
    clean_corpus, V = NGramModel.corpus_preprocessing(corpus)
    n_grams = NGramModel.n_grams(clean_corpus, n)

    self.n = n
    self.V = V
    self.frequency_distribution = NGramModel.frequency_distribution(n_grams)
    self.conditional_probability_distributions = NGramModel.conditional_probability_distributions(self.frequency_distribution, n)

  @staticmethod
  def corpus_preprocessing(corpus):
    clean_corpus = [['<s>'] + [token.lower() for token in sentence if token.isalnum()] + ['</s>'] for sentence in corpus]

    unique_words = set()
    for sentence in clean_corpus:
      unique_words |= set(sentence)

    return clean_corpus, len(unique_words)

  @staticmethod
  def n_grams(corpus, n):
    n_grams = [[] for _ in range(n)]
    for i in range(1, n + 1):
      for sentence in corpus:
        n_grams[i-1] += list(ngrams(sentence, i))
    return n_grams

  @staticmethod
  def frequency_distribution(n_grams):
    return [Counter(n_gram) for n_gram in n_grams]

  @staticmethod
  def conditional_probability_distributions(frequency_distribution, n):
    conditional_probability_distributions = copy.deepcopy(frequency_distribution)

    for i in range(n-1, 0, -1):
      for context in conditional_probability_distributions[i]:
        conditional_probability_distributions[i][context] /= frequency_distribution[i-1][context[:-1]]

    for context in conditional_probability_distributions[0]:
      conditional_probability_distributions[0][context] /= sum(frequency_distribution[0].values())

    return conditional_probability_distributions

  @staticmethod
  def laplace_smoothing(frequency_distribution, context, alpha, n, V):
    if n ==1:
      return (frequency_distribution[0].get(context, 0) + alpha) / (sum(frequency_distribution[0].values()) +  alpha * V)
    return (frequency_distribution[n-1].get(context, 0) + alpha) / (frequency_distribution[n-2].get(context[:-1], 0) +  alpha * V)

  def generate_sentence(self, max_length = 30):
    sentence = "<s>"
    number_of_tokens = 1

    while number_of_tokens < max_length + 1:
      current_n = min(number_of_tokens + 1, self.n)
      conditional_probability_distribution = self.conditional_probability_distributions[current_n - 1]

      sentence_tokens = sentence.split()
      context = tuple(sentence_tokens[-(current_n - 1):]) if current_n != 1 else tuple()

      possible_n_grams = []
      possible_probabilities = []

      for n_gram, prob in conditional_probability_distribution.items():
          if n_gram[:-1] == context:
              possible_n_grams.append(n_gram)
              possible_probabilities.append(prob)

      total_prob = sum(possible_probabilities)
      probabilities = [p / total_prob for p in possible_probabilities]

      chosen_index = np.random.choice(len(possible_n_grams), p=probabilities)
      next_word = possible_n_grams[chosen_index][-1]

      if next_word == '<s>':
        continue

      sentence += ' ' + next_word

      if next_word == '</s>':
        return sentence

      number_of_tokens += 1

    return sentence + ' </s>'

  def evaluation(self, corpus, alpha = 1):
    clean_corpus, _ = NGramModel.corpus_preprocessing(corpus)

    total_log_probability = 0.0
    total_probability_count = 0

    for sentence_index, sentence in enumerate(clean_corpus):
      for end_index in range(2, len(sentence) + 1):
            context_start = max(0, end_index - self.n)

            ngram = tuple(sentence[context_start : end_index])
            n = len(ngram)

            probabilty = NGramModel.laplace_smoothing(self.frequency_distribution, ngram, alpha, n, self.V)

            total_log_probability += math.log(probabilty)
            total_probability_count += 1

    average_log_likelihood = total_log_probability / total_probability_count
    perplexity = math.exp(-average_log_likelihood)

    return perplexity

# **Evaluation**

## **unigram**

In [4]:
unigram = NGramModel(training_corpus, 1)

### **Sentence-Generation**

In [5]:
unigram.generate_sentence()

'<s> publius dying sonne brut did brutus he about sleepe breed may haue your macb he and 2 i shame you magicke haue or let to d </s>'

### **Perplexity**

In [6]:
unigram.evaluation(testing_corpus)

776.0275847175914

## **bigram**

In [34]:
bigram = NGramModel(training_corpus, 2)

### **Sentence-Generation**

In [56]:
bigram.generate_sentence()

'<s> why for your colour but say you for it seem d pretence i an ideot full else that but go in borrowed robes sit i am cabin d tell tale </s>'

### **Perplexity**

In [9]:
bigram.evaluation(testing_corpus)

2421.4330774498176

## **trigram**

In [10]:
trigram = NGramModel(training_corpus, 3)

### **Sentence-Generation**

In [65]:
trigram.generate_sentence()

'<s> that i am glad that my keene knife see not the same dagger for my deere brother this was he not resembled my father a traitor </s>'

### **Perplexity**

In [12]:
trigram.evaluation(testing_corpus)

4053.3568613036573

## **tetragram**

In [13]:
tetragram = NGramModel(training_corpus, 4)

### **Sentence-Generation**

In [66]:
tetragram.generate_sentence()

'<s> shall one of vs that strucke the formost man of all this world but for supporting robbers shall we now contaminate our fingers with base bribes </s>'

### **Perplexity**

In [15]:
tetragram.evaluation(testing_corpus)

4256.271146043696

## **pentagram**

In [16]:
pentagram = NGramModel(training_corpus, 5)

### **Sentence-Generation**

In [38]:
pentagram.generate_sentence()

'<s> i take my leaue of you shall not be long but ile be heere againe things at the worst will cease or else climbe vpward to what they were before </s>'

### **Perplexity**

In [18]:
pentagram.evaluation(testing_corpus)

4268.153032479262

## **hexagram**

In [19]:
hexagram = NGramModel(training_corpus, 6)

### **Sentence-Generation**

In [39]:
hexagram.generate_sentence()

'<s> but gentle heauens cut short all intermission front to front bring thou this fiend of scotland and my selfe within my swords length set him if he scape heauen forgiue </s>'

### **Perplexity**

In [21]:
hexagram.evaluation(testing_corpus)

4269.480531443316

## **heptagram**

In [22]:
heptagram = NGramModel(training_corpus, 7)

### **Sentence-Generation**

In [41]:
heptagram.generate_sentence()

'<s> fla go go good countrymen and for this fault assemble all the poore men of your sort draw them to tyber bankes and weepe your teares into the channell till </s>'

### **Perplexity**

In [24]:
heptagram.evaluation(testing_corpus)

4269.65773925872

## **octagram**

In [67]:
octagram = NGramModel(training_corpus, 8)

### **Sentence-Generation**

In [76]:
octagram.generate_sentence()

'<s> so in the world tis furnish d well with men and men are flesh and blood and apprehensiue yet in the number i do know but one that vnassayleable holds </s>'

### **Perplexity**

In [27]:
octagram.evaluation(testing_corpus)

4269.6576887514975

## **enneagram**

In [28]:
enneagram = NGramModel(training_corpus, 9)

### **Sentence-Generation**

In [80]:
enneagram.generate_sentence()

'<s> strato thou hast bin all this while asleepe farewell to thee to strato countrymen my heart doth ioy that yet in all my life i found no man but he </s>'

### **Perplexity**

In [30]:
enneagram.evaluation(testing_corpus)

4269.6576887514975

## **decagram**

In [31]:
decagram = NGramModel(training_corpus, 10)

### **Sentence-Generation**

In [85]:
decagram.generate_sentence()

'<s> but those that vnderstood him smil d at one another and shooke their heads but for mine owne part it was greeke to me </s>'

### **Perplexity**

In [33]:
decagram.evaluation(testing_corpus)

4269.6576887514975